In [2]:
# ============================================================
# TELECOM CUSTOMER SERVICE MULTI-AGENT SYSTEM
# LangGraph + GPT-4o-mini + Embedding Memory
# Colab Friendly Single File
# ============================================================

# INSTALL DEPENDENCIES
#!pip install -q langgraph langchain langchain-openai faiss-cpu

# ============================================================
# IMPORTS
# ============================================================

import os
import json
import faiss
import numpy as np
from typing import TypedDict, List, Dict

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# ============================================================
# OPENAI API KEY
# ============================================================

# Paste your OpenAI API Key here
os.environ["OPENAI_API_KEY"] = ""

# ============================================================
# LLM + EMBEDDING MODELS
# ============================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# ============================================================
# MEMORY STORE
# ============================================================

memory_texts = []
memory_metadata = []

dimension = 1536
index = faiss.IndexFlatL2(dimension)

# ============================================================
# SAVE MEMORY
# ============================================================

def save_memory(text, metadata={}):
    vector = embedding_model.embed_query(text)

    memory_texts.append(text)
    memory_metadata.append(metadata)

    index.add(np.array([vector]).astype("float32"))

# ============================================================
# SEARCH MEMORY
# ============================================================

def search_memory(query, top_k=3):
    if len(memory_texts) == 0:
        return []

    query_vector = embedding_model.embed_query(query)

    D, I = index.search(
        np.array([query_vector]).astype("float32"),
        top_k
    )

    results = []

    for idx in I[0]:
        if idx < len(memory_texts):
            results.append({
                "text": memory_texts[idx],
                "metadata": memory_metadata[idx]
            })

    return results

# ============================================================
# DUMMY TOOLS
# ============================================================

def create_ticket(issue):
    ticket_id = f"TICK-{1000 + len(memory_texts)}"

    return {
        "ticket_id": ticket_id,
        "status": "OPEN",
        "issue": issue
    }

def check_bill(customer_name):
    return f"{customer_name}'s current bill amount is ₹1299."

def restart_network():
    return "Network restart signal sent successfully."

def check_data_balance():
    return "You have 1.5GB remaining for today."

# ============================================================
# AGENT STATE
# ============================================================

class AgentState(TypedDict):
    user_input: str
    response: str
    agent: str

# ============================================================
# BILLING AGENT
# ============================================================

def billing_agent(state):
    user_input = state["user_input"]

    memories = search_memory(user_input)

    memory_context = "\n".join(
        [m["text"] for m in memories]
    )

    result = check_bill("Customer")

    prompt = f"""
You are Ravi, a Telecom Billing Support Agent.

Role:
- Help with recharge
- Billing
- Payment issues

Previous Memories:
{memory_context}

Tool Result:
{result}

Customer Message:
{user_input}

Give a professional customer support reply.
"""

    response = llm.invoke(prompt).content

    save_memory(
        f"Billing Conversation: {user_input} -> {response}"
    )

    return {
        "response": response,
        "agent": "Billing Agent Ravi"
    }

# ============================================================
# TECHNICAL SUPPORT AGENT
# ============================================================

def technical_agent(state):
    user_input = state["user_input"]

    memories = search_memory(user_input)

    memory_context = "\n".join(
        [m["text"] for m in memories]
    )

    network_result = restart_network()

    ticket = create_ticket(user_input)

    prompt = f"""
You are Priya, a Telecom Technical Support Agent.

Role:
- Internet issues
- Slow network
- Router problems

Previous Memories:
{memory_context}

Tool Results:
{network_result}

Generated Ticket:
{ticket}

Customer Message:
{user_input}

Important:
- Mention the ticket ID
- Remember the ticket
- Sound helpful
"""

    response = llm.invoke(prompt).content

    save_memory(
        f"Technical Ticket {ticket['ticket_id']} for issue: {user_input}"
    )

    save_memory(
        f"Technical Conversation: {user_input} -> {response}",
        metadata={"ticket_id": ticket["ticket_id"]}
    )

    return {
        "response": response,
        "agent": "Technical Agent Priya"
    }

# ============================================================
# SIM + NETWORK AGENT
# ============================================================

def sim_agent(state):
    user_input = state["user_input"]

    memories = search_memory(user_input)

    memory_context = "\n".join(
        [m["text"] for m in memories]
    )

    balance = check_data_balance()

    prompt = f"""
You are Arjun, a Telecom SIM & Network Support Agent.

Role:
- SIM activation
- Data balance
- Network coverage
- 5G support

Previous Memories:
{memory_context}

Tool Result:
{balance}

Customer Message:
{user_input}

Give a useful telecom support response.
"""

    response = llm.invoke(prompt).content

    save_memory(
        f"SIM Conversation: {user_input} -> {response}"
    )

    return {
        "response": response,
        "agent": "SIM Agent Arjun"
    }

# ============================================================
# ROUTER
# ============================================================

def router(state):
    text = state["user_input"].lower()

    if any(word in text for word in [
        "bill",
        "payment",
        "recharge",
        "invoice"
    ]):
        return "billing"

    elif any(word in text for word in [
        "internet",
        "slow",
        "wifi",
        "network",
        "router",
        "issue",
        "not working"
    ]):
        return "technical"

    else:
        return "sim"

# ============================================================
# BUILD LANGGRAPH
# ============================================================

workflow = StateGraph(AgentState)

workflow.add_node("billing", billing_agent)
workflow.add_node("technical", technical_agent)
workflow.add_node("sim", sim_agent)

workflow.set_conditional_entry_point(
    router,
    {
        "billing": "billing",
        "technical": "technical",
        "sim": "sim"
    }
)

workflow.add_edge("billing", END)
workflow.add_edge("technical", END)
workflow.add_edge("sim", END)

app = workflow.compile()




 TELECOM CUSTOMER SERVICE MULTI-AGENT SYSTEM 
Type 'exit' to stop.

YOU: Hi

[SIM Agent Arjun]
Hello! Thank you for reaching out. How can I assist you today? If you need help with SIM activation, checking your data balance, network coverage, or 5G support, just let me know! Currently, you have 1.5GB of data remaining for today. If you have any specific questions or issues, feel free to share!

---------------------------------------------------

YOU: Data Balance

[SIM Agent Arjun]
Hello! Thank you for reaching out. Currently, you have 1.5GB of data remaining for today. If you need assistance with anything else, such as SIM activation, network coverage, or 5G support, just let me know!

---------------------------------------------------

YOU: exit

Goodbye 👋

================ MEMORY STORE ================

1. SIM Conversation: Hi -> Hello! Thank you for reaching out. How can I assist you today? If you need help with SIM activation, checking your data balance, network coverage, or 5G 

In [3]:
# ============================================================
# INTERACTIVE CHAT LOOP
# ============================================================

print("\n===================================================")
print(" TELECOM CUSTOMER SERVICE MULTI-AGENT SYSTEM ")
print("===================================================")
print("Type 'exit' to stop.\n")

while True:

    user_query = input("YOU: ")

    if user_query.lower() == "exit":
        print("\nGoodbye 👋")
        break

    result = app.invoke({
        "user_input": user_query
    })

    print(f"\n[{result['agent']}]")
    print(result["response"])
    print("\n---------------------------------------------------\n")

# ============================================================
# OPTIONAL: VIEW MEMORY
# ============================================================

print("\n================ MEMORY STORE ================\n")

for i, memory in enumerate(memory_texts):
    print(f"{i+1}. {memory}")


 TELECOM CUSTOMER SERVICE MULTI-AGENT SYSTEM 
Type 'exit' to stop.

YOU: Hi

[SIM Agent Arjun]
Hello! Thank you for reaching out. How can I assist you today? If you need help with SIM activation, checking your data balance, network coverage, or 5G support, just let me know! Currently, you have 1.5GB of data remaining for today. If there's anything specific you'd like assistance with, feel free to share!

---------------------------------------------------

YOU: exit

Goodbye 👋

================ MEMORY STORE ================

1. SIM Conversation: Hi -> Hello! Thank you for reaching out. How can I assist you today? If you need help with SIM activation, checking your data balance, network coverage, or 5G support, just let me know! Currently, you have 1.5GB of data remaining for today. If you have any specific questions or issues, feel free to share!
2. SIM Conversation: Data Balance -> Hello! Thank you for reaching out. Currently, you have 1.5GB of data remaining for today. If you need a

In [4]:
# ============================================================
# AUTOMATED TESTING SECTION
# Different Queries + Memory Validation
# ============================================================

test_queries = [

    # ----------------------------------------
    # BILLING TESTS
    # ----------------------------------------

    "My bill amount is too high this month",

    "Recharge was done but not updated",

    "Can you check my payment status?",

    # ----------------------------------------
    # TECHNICAL TESTS
    # ----------------------------------------

    "My internet is not working",

    "WiFi is very slow",

    "Router keeps disconnecting",

    # ----------------------------------------
    # MEMORY TESTS
    # ----------------------------------------

    "What is my ticket ID?",

    "What issue did I report earlier?",

    "Can you summarize my previous complaint?",

    # ----------------------------------------
    # SIM + NETWORK TESTS
    # ----------------------------------------

    "How much data balance do I have?",

    "My SIM card is not activating",

    "Is 5G available in Chennai?",

    # ----------------------------------------
    # MULTI TURN CONTEXT TEST
    # ----------------------------------------

    "I still have the same internet problem",

    "Did you already create a ticket for me?",

    "Can you check previous network issue?"
]

# ============================================================
# RUN TESTS
# ============================================================

print("\n")
print("=" * 70)
print(" RUNNING TELECOM AGENT TEST CASES ")
print("=" * 70)

for i, query in enumerate(test_queries):

    print(f"\nTEST CASE {i+1}")
    print("-" * 70)

    print(f"\nUSER:")
    print(query)

    result = app.invoke({
        "user_input": query
    })

    print(f"\nAGENT:")
    print(result["agent"])

    print(f"\nRESPONSE:")
    print(result["response"])

    print("\n" + "=" * 70)

# ============================================================
# MEMORY VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print(" MEMORY VALIDATION ")
print("=" * 70)

memory_search_queries = [

    "ticket id",
    "internet issue",
    "billing complaint",
    "SIM issue"
]

for query in memory_search_queries:

    print(f"\nSearching Memory For: {query}")
    print("-" * 50)

    memories = search_memory(query)

    if len(memories) == 0:
        print("No memories found")

    else:
        for idx, mem in enumerate(memories):

            print(f"\nMemory {idx+1}:")
            print(mem["text"])

# ============================================================
# COMPLETE MEMORY STORE
# ============================================================

print("\n")
print("=" * 70)
print(" COMPLETE MEMORY STORE ")
print("=" * 70)

for i, mem in enumerate(memory_texts):

    print(f"\n{i+1}. {mem}")

# ============================================================
# TEST SPECIFIC TICKET RECALL
# ============================================================

print("\n")
print("=" * 70)
print(" TICKET RECALL TEST ")
print("=" * 70)

ticket_test = app.invoke({
    "user_input": "Can you tell me my previously generated ticket number?"
})

print(ticket_test["response"])

# ============================================================
# STRESS TEST
# ============================================================

print("\n")
print("=" * 70)
print(" STRESS TEST ")
print("=" * 70)

stress_queries = [
    "internet problem",
    "bill issue",
    "network slow",
    "payment failed",
    "SIM not working"
]

for i in range(10):

    q = stress_queries[i % len(stress_queries)]

    result = app.invoke({
        "user_input": q
    })

    print(f"\nIteration {i+1}")
    print(f"Query: {q}")
    print(f"Agent: {result['agent']}")

print("\n")
print("=" * 70)
print(" ALL TESTS COMPLETED ")
print("=" * 70)



 RUNNING TELECOM AGENT TEST CASES 

TEST CASE 1
----------------------------------------------------------------------

USER:
My bill amount is too high this month

AGENT:
Billing Agent Ravi

RESPONSE:
Hello! Thank you for reaching out. I understand that you're concerned about your bill amount of ₹1299 this month. Let’s take a closer look at the charges to see if we can identify any discrepancies or provide clarification.

Could you please provide me with more details about your usage this month? For example, any specific services you used frequently, or if there were any unexpected charges you noticed? This will help me assist you better. Thank you!


TEST CASE 2
----------------------------------------------------------------------

USER:
Recharge was done but not updated

AGENT:
Billing Agent Ravi

RESPONSE:
Hello! Thank you for reaching out. I understand that you've completed a recharge, but it hasn't been updated in your account yet. I apologize for any inconvenience this may ha